In [1]:
from tensorflow.keras.utils import get_file

ratings_train_path = get_file('ratings_train.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt')
ratings_test_path = get_file('ratings_test.txt', 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt')

print(ratings_train_path)
print(ratings_test_path)

/root/.keras/datasets/ratings_train.txt
/root/.keras/datasets/ratings_test.txt


In [2]:
import pandas as pd

ratings_train_df = pd.read_csv(ratings_train_path, sep='\t')
ratings_test_df = pd.read_csv(ratings_test_path, sep='\t')

In [3]:
ratings_train_df.isnull().sum(), ratings_test_df.isnull().sum()

(id          0
 document    5
 label       0
 dtype: int64,
 id          0
 document    3
 label       0
 dtype: int64)

In [4]:
ratings_train_df = ratings_train_df.dropna(how='any')
ratings_test_df = ratings_test_df.dropna(how='any')

In [5]:
ratings_train_df = ratings_train_df.sample(n=15000, random_state=0)
ratings_test_df = ratings_test_df.sample(n=5000, random_state=0)

ratings_train_df['label'].value_counts(), ratings_test_df['label'].value_counts()

(label
 0    7512
 1    7488
 Name: count, dtype: int64,
 label
 0    2532
 1    2468
 Name: count, dtype: int64)

In [ ]:
X_train = ratings_train_df['document'].values.tolist()
y_train = ratings_train_df['label'].values.tolist()
X_test = ratings_test_df['document'].values.tolist()
y_test = ratings_test_df['label'].values.tolist()

In [7]:
from transformers import AutoModel, AutoTokenizer
from transformers import BertForSequenceClassification

model_name = 'klue/bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [8]:
X_train = tokenizer(X_train, padding=True, truncation=True, return_tensors='pt')
X_test = tokenizer(X_test, padding=True, truncation=True, return_tensors='pt')

X_train[:3], X_test[:3]

([Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=142, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])],
 [Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
  Encoding(num_tokens=118, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])])

In [9]:
print(X_train['input_ids'][0])
print(X_train['attention_mask'][0])
print(X_train['token_type_ids'][0])

tensor([    2,  1800,  2178,   860,  3629, 16516,  2031,    18,    18,    18,
        14242,  2205,  2062,     3,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
train_dataset = NSMCDataset(X_train, y_train)
test_dataset = NSMCDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [12]:
import torch
from transformers import get_scheduler

epochs = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_training_steps = len(train_dataloader) * epochs
num_warmup_steps = int(num_training_steps * 0.1)

lr_scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [ ]:
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)

for epoch in tqdm(range(epochs)):
    model.train()

    total_loss = 0

    for batch in tqdm(train_dataloader, desc=f'{epoch+1}/{epochs}', leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        lr_scheduler.step()

        total_loss += loss.item()

    print(total_loss / len(train_dataloader))

cuda


  0%|          | 0/5 [00:00<?, ?it/s]

1/5:   0%|          | 0/235 [00:00<?, ?it/s]

2/5:   0%|          | 0/235 [00:00<?, ?it/s]

3/5:   0%|          | 0/235 [00:00<?, ?it/s]

4/5:   0%|          | 0/235 [00:00<?, ?it/s]

5/5:   0%|          | 0/235 [00:00<?, ?it/s]

In [14]:
model.save_pretrained('nsmc_model/bert-base')
tokenizer.save_pretrained('nsmc_model/bert-base')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('nsmc_model/bert-base/tokenizer_config.json',
 'nsmc_model/bert-base/tokenizer.json')

In [15]:
from transformers import TextClassificationPipeline

sentiment_classifier = TextClassificationPipeline(
    tokenizer=tokenizer,
    model=model,
    framework='pt',
    tok_k=None
)

In [27]:
model.config.id2label = {
    0: '부정',
    1: '긍정',
}

In [28]:
sentiment_classifier('이것은 제 생에 가장 훌륭한 영화입니다! 대단합니다.')

[{'label': '부정', 'score': 0.9911638498306274}]

In [29]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = getpass("HF_TOKEN: ")
login(token=HF_TOKEN)

LocalProtocolError: Illegal header value b'Bearer '

In [23]:
REPO_NAME = 'bert-base-nsmc'

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...op1t9b0/model.safetensors:   4%|3         | 16.0MB /  442MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/kty2001/bert-base-nsmc/commit/a697ecd5a97d2b2bbfbe6c43a27900dd1cde5f14', commit_message='Upload tokenizer', commit_description='', oid='a697ecd5a97d2b2bbfbe6c43a27900dd1cde5f14', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kty2001/bert-base-nsmc', endpoint='https://huggingface.co', repo_type='model', repo_id='kty2001/bert-base-nsmc'), pr_revision=None, pr_num=None)

In [24]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

HUB_NAME = 'kty2001/bert-base-nsmc'

tokenizer = AutoTokenizer.from_pretrained(HUB_NAME)
model = AutoModelForSequenceClassification.from_pretrained(HUB_NAME)

config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/404 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [25]:
from transformers import pipeline

sentiment_classifier = pipeline('text-classification', model=HUB_NAME)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [31]:
sentiment_classifier([
    '한국 영화는 이래서 안돼~',
    '역시 봉감독이 최고야!',
    '진짜 정말고 강하게 재미없다.',
    'godgod'
])

[{'label': '긍정', 'score': 0.9993119239807129},
 {'label': '부정', 'score': 0.9765624403953552},
 {'label': '긍정', 'score': 0.9998571872711182},
 {'label': '부정', 'score': 0.992493748664856}]